In [36]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD

In [37]:
ratings = pd.read_csv('../Mini_trabalho_7/dados_processados/ratings_processado.csv')
movies  = pd.read_csv('../Mini_trabalho_7/dados_processados/movies_processado.csv')

media_global = ratings['rating'].mean()

matriz = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(media_global)

svd = TruncatedSVD(n_components=10, random_state=42)
matriz_reduzida = svd.fit_transform(matriz)
df_reconstruida = pd.DataFrame(svd.inverse_transform(matriz_reduzida),
                                index=matriz.index, columns=matriz.columns)

print('Modelo treinado!')

Modelo treinado!


In [38]:
def recomendar(user_id, n_recomendacoes=10):
    if user_id not in df_reconstruida.index:
        print(f'Usuário {user_id} não encontrado.')
        return
    
    # Filmes que o usuário já avaliou
    ja_assistidos = ratings[ratings['userId'] == user_id]['movieId'].values
    
    # Notas previstas pra todos os filmes
    notas_previstas = df_reconstruida.loc[user_id]
    
    # Remove filmes já assistidos
    notas_previstas = notas_previstas[~notas_previstas.index.isin(ja_assistidos)]
    
    # Top N filmes
    top_filmes = notas_previstas.nlargest(n_recomendacoes).index
    
    # Busca títulos
    filmes_originais = pd.read_csv('../Mini_trabalho_4/dados_processados/movies_processado.csv')
    recomendados = filmes_originais[filmes_originais['movieId'].isin(top_filmes)][['movieId', 'title']]
    
    print(f'Recomendações para o usuário {user_id}:\n')
    for i, (_, row) in enumerate(recomendados.iterrows(), 1):
        print(f'{i}. {row["title"]}')


In [39]:
import matplotlib.pyplot as plt

def perfil_usuario(user_id):
    avaliacoes = ratings[ratings['userId'] == user_id]
    
    if avaliacoes.empty:
        print(f'Usuário {user_id} não encontrado.')
        return
    
    filmes_avaliados = movies[movies['movieId'].isin(avaliacoes['movieId'])]
    generos = filmes_avaliados.drop(columns=['movieId', 'title']).sum().sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(8, 4))
    generos.plot(kind='barh', color='steelblue')
    plt.title(f'Gêneros favoritos — Usuário {user_id}', fontweight='bold')
    plt.xlabel('Quantidade de filmes assistidos')
    plt.tight_layout()
    plt.savefig(f'assets/perfil_usuario_{user_id}.png')
    plt.show()


In [40]:
import ipywidgets as widgets
from IPython.display import display, clear_output

output = widgets.Output()

user_input = widgets.IntText(
    value=1,
    description='Usuário ID:',
    style={'description_width': 'initial'}
)

botao = widgets.Button(description='Recomendar', button_style='primary')

def ao_clicar(b):
    with output:
        clear_output()
        user_id = user_input.value
        
        if user_id < 1 or user_id > 610:
            print(f'Usuário {user_id} não existe. Digite um ID entre 1 e 610.')
            return
        
        print(f'=== Perfil do Usuário {user_id} ===\n')
        perfil_usuario(user_id)
        print(f'\n=== Recomendações para o Usuário {user_id} ===\n')
        recomendar(user_id)
botao.on_click(ao_clicar)

display(user_input, botao, output)

IntText(value=1, description='Usuário ID:', style=DescriptionStyle(description_width='initial'))

Button(button_style='primary', description='Recomendar', style=ButtonStyle())

Output()

In [41]:
def recomendar_novo_usuario(generos_favoritos, epoca=None, popularidade=None, n_recomendacoes=10):
    # Filtra por gênero
    mask = movies[generos_favoritos].any(axis=1)
    filmes_genero = movies[mask][['movieId', 'title']]
    
    # Junta com ratings
    stats = ratings.groupby('movieId')['rating'].agg(media='mean', contagem='count').reset_index()
    filmes_com_stats = filmes_genero.merge(stats, on='movieId')
    
    # Filtra por época
    if epoca == 'antigos':
        filmes_com_stats = filmes_com_stats[filmes_com_stats['title'].str.extract(r'\((\d{4})\)')[0].astype(float) < 1980]
    elif epoca == '80_90':
        ano = filmes_com_stats['title'].str.extract(r'\((\d{4})\)')[0].astype(float)
        filmes_com_stats = filmes_com_stats[(ano >= 1980) & (ano <= 1999)]
    elif epoca == 'recentes':
        filmes_com_stats = filmes_com_stats[filmes_com_stats['title'].str.extract(r'\((\d{4})\)')[0].astype(float) >= 2000]
    
    # Filtra por popularidade
    if popularidade == 'populares':
        filmes_com_stats = filmes_com_stats[filmes_com_stats['contagem'] >= 50]
    elif popularidade == 'nicho':
        filmes_com_stats = filmes_com_stats[filmes_com_stats['contagem'] < 50]
    else:
        filmes_com_stats = filmes_com_stats[filmes_com_stats['contagem'] >= 10]
    
    top = filmes_com_stats.sort_values('media', ascending=False).head(n_recomendacoes)
    
    print('Recomendações baseadas no seu perfil:\n')
    for i, (_, row) in enumerate(top.iterrows(), 1):
        print(f'{i}. {row["title"]} (★ {row["media"]:.2f})')

In [44]:
output2 = widgets.Output()

checkboxes = [widgets.Checkbox(value=False, description=g) for g in generos_disponiveis]
grid = widgets.GridBox(checkboxes, layout=widgets.Layout(grid_template_columns='repeat(3, 200px)'))

epoca_widget = widgets.RadioButtons(
    options=[('Qualquer época', None), ('Antes de 1980', 'antigos'), ('Anos 80/90', '80_90'), ('Anos 2000+', 'recentes')],
    description='Época:',
    style={'description_width': 'initial'}
)

popularidade_widget = widgets.RadioButtons(
    options=[('Qualquer', None), ('Populares', 'populares'), ('Menos conhecidos', 'nicho')],
    description='Popularidade:',
    style={'description_width': 'initial'}
)

botao2 = widgets.Button(description='Recomendar para mim', button_style='success')

def ao_clicar_novo(b):
    with output2:
        clear_output()
        selecionados = [cb.description for cb in checkboxes if cb.value]
        if not selecionados:
            print('Selecione pelo menos um gênero.')
            return
        print(f'Gêneros: {", ".join(selecionados)}')
        print(f'Época: {epoca_widget.value} | Popularidade: {popularidade_widget.value}\n')
        recomendar_novo_usuario(selecionados, epoca_widget.value, popularidade_widget.value)

botao2.on_click(ao_clicar_novo)

print('Selecione seus gêneros favoritos:')
display(grid, epoca_widget, popularidade_widget, botao2, output2)

Selecione seus gêneros favoritos:


GridBox(children=(Checkbox(value=False, description='Action'), Checkbox(value=False, description='Adventure'),…

RadioButtons(description='Época:', options=(('Qualquer época', None), ('Antes de 1980', 'antigos'), ('Anos 80/…

RadioButtons(description='Popularidade:', options=(('Qualquer', None), ('Populares', 'populares'), ('Menos con…

Button(button_style='success', description='Recomendar para mim', style=ButtonStyle())

Output()